<center>
<a href="https://www.umontpellier.fr/"><img src="https://www.umontpellier.fr/wp-content/uploads/2022/10/logo_um_2022_rouge_rvb.svg" width="200"/></a>&nbsp;&nbsp;
<a href="https://economie.edu.umontpellier.fr/"><img src="https://economie.edu.umontpellier.fr/files/2014/12/economie_rvb_2015-300x137.png" width="160"/></a>
</center>

<div align="center">

#  Phase 1 — Nettoyage et Validation de la Base `dossier.csv`

| Nom et Prénom | Rôle |
|---|---|
| Randriamisaina Tsiory-Fanomezana | Membre de l'équipe |
| SHIRALI POUR Amir | Membre de l'équipe |

</div>

---
## Objectif de cette phase

Ce notebook constitue la **première phase** du pipeline de traitement des données.  
Il applique, dans l'ordre, toutes les corrections décrites dans [`docs/phase1_dossier.md`](../docs/phase1_dossier.md).

Pour chaque anomalie, la démarche est systématique :
1. **Constat** — quantification du problème
2. **Décision** — choix documenté avec justification métier/statistique  
3. **Action** — code de correction
4. **Vérification** — preuve que la correction est effective


---
## Section 0 — Initialisation

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# --- Localisation de la racine du projet
def localiser_racine_du_projet():
    """Remonte l'arborescence jusqu'à trouver la racine du projet (présence de .git ou requirements.txt)."""
    repertoire_courant = Path.cwd()
    while True:
        if any((repertoire_courant / m).exists() for m in ['.git', 'requirements.txt']):
            break
        if repertoire_courant.parent == repertoire_courant:
            break
        repertoire_courant = repertoire_courant.parent
    if Path.cwd() != repertoire_courant:
        os.chdir(repertoire_courant)
    return repertoire_courant.resolve()

REPERTOIRE_RACINE = localiser_racine_du_projet()
sys.path.insert(0, str(REPERTOIRE_RACINE / 'src'))

from utils.dataframe_styler import style_duplicates
print(f" Racine du projet : {REPERTOIRE_RACINE}")

In [ ]:
# --- Chargement de la base brute
dossier_original = pd.read_csv('data/cleaned_dossier.csv', encoding='utf-8', sep=',', quotechar='"')

# Copie de travail — on ne touche JAMAIS à dossier_original
dossier = dossier_original.copy()

print(f" Dimensions chargées : {dossier.shape[0]:,} lignes × {dossier.shape[1]} colonnes")
print(f" Identifiants uniques : {dossier['Numero_dossier_ID'].nunique():,}")
print(f" Identifiants dupliqués : {dossier['Numero_dossier_ID'].duplicated().sum():,}")

---
## Section 1 — Vue d'ensemble de la qualité des données

### 1.1 Types de données et valeurs non-nulles

In [ ]:
dossier.info()